In [48]:
import warnings
warnings.filterwarnings("ignore")
from yfinance import download

In [49]:
ticker = 'klbn4.sa'

In [50]:
df = download(ticker, period="max", interval="1mo", auto_adjust=False).droplevel(1, axis=1)

[*********************100%***********************]  1 of 1 completed


In [51]:
df

Price,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2008-01-01,0.547920,1.096309,1.202520,0.936093,1.175517,186257479
2008-02-01,0.514630,1.029702,1.123312,1.029702,1.101710,153146900
2008-03-01,0.521828,1.044104,1.069306,0.946894,1.044104,147539128
2008-04-01,0.584808,1.170117,1.261926,1.033303,1.038703,174480878
2008-05-01,0.578509,1.157515,1.312331,1.157515,1.170117,117099398
...,...,...,...,...,...,...
2025-09-01,3.328085,3.544554,3.742574,3.524752,3.663366,59051165
2025-10-01,3.337381,3.554455,3.623762,3.415841,3.564356,84067249
2025-11-01,3.309492,3.524752,3.712871,3.435643,3.564356,82341664


In [52]:
from numpy import where
df['tipo'] = where(df['Close'] > df['Open'], 'alta', 'baixa')

In [53]:
df['sombra_superior'] = where(df['tipo'] == 'alta', 
                              df['High'] - df['Close'],
                              df['High'] - df['Open'])

In [54]:
df['sombra_inferior'] = where(df['tipo'] == 'baixa',
                              df['Close'] - df['Low'],
                              df['Open'] - df['Low'])

In [55]:
df

Price,Adj Close,Close,High,Low,Open,Volume,tipo,sombra_superior,sombra_inferior
Date,,,,,,,,,
2008-01-01,0.547920,1.096309,1.202520,0.936093,1.175517,186257479,baixa,0.027003,0.160216
2008-02-01,0.514630,1.029702,1.123312,1.029702,1.101710,153146900,baixa,0.021602,0.000000
2008-03-01,0.521828,1.044104,1.069306,0.946894,1.044104,147539128,baixa,0.025202,0.097210
2008-04-01,0.584808,1.170117,1.261926,1.033303,1.038703,174480878,alta,0.091809,0.005400
2008-05-01,0.578509,1.157515,1.312331,1.157515,1.170117,117099398,baixa,0.142214,0.000000
...,...,...,...,...,...,...,...,...,...
2025-09-01,3.328085,3.544554,3.742574,3.524752,3.663366,59051165,baixa,0.079208,0.019802
2025-10-01,3.337381,3.554455,3.623762,3.415841,3.564356,84067249,baixa,0.059406,0.138614
2025-11-01,3.309492,3.524752,3.712871,3.435643,3.564356,82341664,baixa,0.148515,0.089109


In [56]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Candlestick(
    x=df.index,
    open=df['Open'],
    high=df['High'],
    low=df['Low'],
    close=df['Close'],
    name='Preço'
))

fig.add_trace(go.Bar(
    x=df.index,
    y=df['sombra_superior'],
    name='Sombra Superior',
    marker=dict(
        color='rgba(0, 102, 204, 0.5)'  # azul com transparência
    ),
    yaxis='y2'
))

fig.add_trace(go.Bar(
    x=df.index,
    y=df['sombra_inferior'],
    name='Sombra Inferior',
    marker=dict(
        color='rgba(0, 102, 204, 0.5)'
    ),
    yaxis='y2'
))

fig.update_layout(
    width=1600,
    height=900,
    font=dict(size=14),
    margin=dict(l=60, r=60, t=60, b=60),
    yaxis2=dict(
        overlaying='y',
        side='right',
        showgrid=False,
        title='Amplitude da Sombra'
    )
)




---

se sequência de 2 meses de alta qual a amplitude da sombra em relação ao mês anterior

In [57]:
df_a1 = df.copy()

# Se 2 dias de alta então tirar o percentual da sombra atualem relação a amplitude do dia anteior
df_a1['a1'] = where((df['tipo'].shift(1) == 'alta') & (df['tipo'] == 'alta'),  
                    df['sombra_inferior'] / (df['Close'].shift(1) - df['Open'].shift(1)),  'NaN')

df_a1.tail(30)

Price,Adj Close,Close,High,Low,Open,Volume,tipo,sombra_superior,sombra_inferior,a1
Date,,,,,,,,,,
2023-08-01,3.437289,4.131413,4.302430,3.969396,4.167416,64422880,baixa,0.135014,0.162017,NaN
2023-09-01,3.624263,4.311431,4.401440,3.960396,4.149414,42521407,alta,0.090009,0.189018,NaN
2023-10-01,3.245947,3.861386,4.401440,3.843384,4.311431,37300039,baixa,0.090009,0.018002,NaN
2023-11-01,3.465410,4.068406,4.239423,3.834383,3.861386,38632904,alta,0.171017,0.027003,NaN
2023-12-01,3.381075,3.969396,4.059405,3.681368,4.032403,78061853,baixa,0.027002,0.288028,NaN
2024-01-01,3.297290,3.843384,4.086408,3.834383,3.969396,44341332,baixa,0.117012,0.009001,NaN
2024-02-01,3.482617,4.059405,4.095409,3.699369,3.843384,42225324,alta,0.036004,0.144015,NaN
2024-03-01,3.915782,4.527452,4.554455,3.906390,4.059405,36672767,alta,0.027003,0.153015,0.7083340231354098
2024-04-01,3.565464,4.122412,4.572457,4.122412,4.536453,30385176,baixa,0.036004,0.000000,NaN


In [58]:
df_a1[df_a1['a1'] != "NaN"]

Price,Adj Close,Close,High,Low,Open,Volume,tipo,sombra_superior,sombra_inferior,a1
Date,,,,,,,,,,
2009-01-01,0.314896,0.630063,0.630063,0.558055,0.613861,49859456,alta,0.000000,0.055806,15.501672240802675
2009-05-01,0.291504,0.583258,0.676867,0.556255,0.556255,272981028,alta,0.093609,0.000000,0.0
2009-09-01,0.377876,0.756075,0.792079,0.651665,0.651665,269078640,alta,0.036004,0.000000,0.0
2009-12-01,0.477743,0.955895,0.981098,0.867686,0.883888,252655284,alta,0.025203,0.016202,0.12162410065504528
2010-07-01,0.466412,0.921692,0.946894,0.865886,0.900090,146521236,alta,0.025202,0.034204,2.3749529229177444
2010-11-01,0.458705,0.882088,0.900090,0.797479,0.855085,401539281,alta,0.018002,0.057606,15.997169577091782
2010-12-01,0.547638,1.053105,1.071107,0.891089,0.891089,470521271,alta,0.018002,0.000000,0.0
2011-03-01,0.617848,1.188118,1.188118,1.060306,1.107110,178812072,alta,0.000000,0.046804,0.5416635050624756
2011-11-01,0.704421,1.299729,1.310531,1.083708,1.083708,331360189,alta,0.010802,0.000000,0.0


In [59]:
df_a1[df_a1['a1'] != "NaN"]['a1'].astype(float)

Date
2009-01-01    15.501672
2009-05-01     0.000000
2009-09-01     0.000000
2009-12-01     0.121624
2010-07-01     2.374953
2010-11-01    15.997170
2010-12-01     0.000000
2011-03-01     0.541664
2011-11-01     0.000000
2011-12-01     0.283334
2012-01-01     0.414284
2012-02-01     5.727261
2012-09-01     3.000118
2012-10-01     0.040816
2012-11-01     0.607410
2012-12-01     6.889128
2013-01-01     0.499999
2013-11-01     3.349985
2013-12-01     2.476264
2014-01-01     1.846172
2014-08-01     0.857140
2014-09-01     2.333333
2014-10-01     1.555550
2014-11-01     0.374990
2014-12-01     0.407406
2015-03-01     0.022222
2016-10-01     0.733331
2017-05-01     1.866672
2017-09-01     0.307694
2017-10-01     0.375000
2018-04-01     0.470596
2018-05-01     4.200058
2019-10-01     3.250026
2019-11-01     0.181820
2019-12-01     0.000000
2020-01-01     0.000000
2020-05-01     0.848486
2020-06-01     0.578948
2020-07-01     0.866667
2020-08-01     5.000000
2020-11-01     6.999859
2020-12-01 

In [60]:
df_a1[df_a1['a1'] != "NaN"]['a1'].astype(float).mean()

np.float64(1.8887408256042941)

In [61]:
len(df_a1[df_a1['a1'] != "NaN"]) / len(df)

0.2350230414746544